In this notebook, <br/>
1] We employed tabnet on subsets (subcategories; swearwords with subcategories (SW); nonswearwords with subcategories (NSW))
2] We employed CatBoost on subsets (subcategories; swearwords with subcategories (SW); nonswearwords with subcategories (NSW)) <br/>
3] We employed CatBoost + SHAP on subsets (subcategories; swearwords with subcategories (SW); nonswearwords with subcategories (NSW)) <br/>
4] Two functions were created: (a) one for executing the TabNet model and (b) one for visualizing the TabNet chart, allowing all subsets (subcategories, SW, and NSW) to be executed with a single call.  <br/>
5] Two functions were created: (a) one for executing the CatBoost model and (b) one for visualizing the CatBoost chart, allowing all subsets (subcategories, SW, and NSW) to be executed with a single call. <br/>
6] Created function of SHAP, allowing all subsets (subcategories, SW, and NSW) to be executed with a single call. <br/>
7] Computed Spearman rank correlation between tabnet and catboost, SHAP on the above subsets <br/>
8] Computed kendall tau between tabnet and catboost, SHAP on the above subset [Extra] <br/>
9] Used lexicon downloaded from published zenodo

**Employ the FI-RC framework on the Hasoc 2020 dataset using Lexicon 2020**

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix,f1_score, precision_score, recall_score

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

Reading the Dataset

In [ ]:
train_df=pd.read_csv("dataset_meta/hindi2020_SW_metadata.csv")
train_df= train_df.dropna()

**in this code, we removed explicit_cricket_personalities because its tweet count=0 and implicit hindus because its tweet count=1**

In [ ]:

#------------------------------------ Subcategories------------------------------

X_features_subcat=train_df[["Explicit_Bollywood_Personalities","Explicit_Cricket_Personalities","Explicit_Historical_Figures","Explicit_Media_Personalities","Explicit_Other_Personalities","Explicit_Politician","Explicit_Caste_Groups","Explicit_Political_Groups","Explicit_Religious_Groups","Explicit_Other_Groups","Implicit_Hindus","Implicit_Muslims","Implicit_Others","Implicit_Personalised_Political_Slurs","Implicit_Political_Group_Slurs"]]


#----------------------------Additional Metadata with swear words subcategories ---------------------------------------------
X_features_swear=train_df[['Swear_Explicit_Bollywood_Personalities','Swear_Explicit_Cricket_Personalities','Swear_Explicit_Historical_Figures','Swear_Explicit_Media_Personalities', 'Swear_Explicit_Other_Personalities', 'Swear_Explicit_Politician','Swear_Explicit_Caste_Groups', 'Swear_Explicit_Religious_Groups',
       'Swear_Explicit_Political_Groups', 'Swear_Explicit_Other_Groups',
       'Swear_Implicit_Hindus', 'Swear_Implicit_Muslims',
       'Swear_Implicit_Others', 'Swear_Implicit_Personalised_Political_Slurs',
       'Swear_Implicit_Political_Group_Slurs']]

X_features_swear.columns = [
    col.replace('Swear_', '') + '_SW'
    for col in X_features_swear.columns
]

#----------------------------Additional Metadata with nonswear words subcategories ---------------------------------------------
X_features_nonswear=train_df[['NonSwear_Explicit_Bollywood_Personalities','NonSwear_Explicit_Cricket_Personalities','NonSwear_Explicit_Historical_Figures','NonSwear_Explicit_Media_Personalities', 'NonSwear_Explicit_Other_Personalities', 'NonSwear_Explicit_Politician','NonSwear_Explicit_Caste_Groups', 'NonSwear_Explicit_Religious_Groups',
       'NonSwear_Explicit_Political_Groups', 'NonSwear_Explicit_Other_Groups',
       'NonSwear_Implicit_Hindus', 'NonSwear_Implicit_Muslims',
       'NonSwear_Implicit_Others', 'NonSwear_Implicit_Personalised_Political_Slurs',
       'NonSwear_Implicit_Political_Group_Slurs']]

X_features_nonswear.columns = [
    col.replace('NonSwear_', '') + '_NSW'
    for col in X_features_nonswear.columns
]


target=train_df["Mismatch"]

In [ ]:
X_features_swear.columns

In [ ]:
X_features_nonswear.columns

In [ ]:
group_map_sub = {
    "Explicit_Caste_Groups": "Explicit Target Groups",
    "Explicit_Other_Groups": "Explicit Target Groups",
    "Explicit_Religious_Groups": "Explicit Target Groups",
    "Explicit_Political_Groups": "Explicit Target Groups",

    "Implicit_Personalised_Political_Slurs": "Implicit Target",
    "Implicit_Hindus": "Implicit Target",
    "Implicit_Political_Group_Slurs": "Implicit Target",
    "Implicit_Others":"Implicit Target",
    "Implicit_Muslims":"Implicit Target",
    
    "Explicit_Cricket_Personalities": "Explicit Target Personalities",
    "Explicit_Media_Personalities": "Explicit Target Personalities",
    "Explicit_Bollywood_Personalities": "Explicit Target Personalities",
    "Explicit_Historical_Figures": "Explicit Target Personalities",
    "Explicit_Other_Personalities": "Explicit Target Personalities",
    "Explicit_Politician": "Explicit Target Personalities"
}

In [ ]:
group_map_swear = {
    "Explicit_Caste_Groups_SW": "Explicit Target Groups",
    "Explicit_Other_Groups_SW": "Explicit Target Groups",
    "Explicit_Religious_Groups_SW": "Explicit Target Groups",
    "Explicit_Political_Groups_SW": "Explicit Target Groups",

    "Implicit_Personalised_Political_Slurs_SW": "Implicit Target",
    "Implicit_Hindus_SW": "Implicit Target",
    "Implicit_Political_Group_Slurs_SW": "Implicit Target",
    "Implicit_Others_SW":"Implicit Target",
    "Implicit_Muslims_SW":"Implicit Target",
    
    "Explicit_Cricket_Personalities_SW": "Explicit Target Personalities",
    "Explicit_Media_Personalities_SW": "Explicit Target Personalities",
    "Explicit_Bollywood_Personalities_SW": "Explicit Target Personalities",
    "Explicit_Historical_Figures_SW": "Explicit Target Personalities",
    "Explicit_Other_Personalities_SW": "Explicit Target Personalities",
    "Explicit_Politician_SW": "Explicit Target Personalities"
}


In [ ]:
group_map_nonswear = {
    "Explicit_Caste_Groups_NSW": "Explicit Target Groups",
    "Explicit_Other_Groups_NSW": "Explicit Target Groups",
    "Explicit_Religious_Groups_NSW": "Explicit Target Groups",
    "Explicit_Political_Groups_NSW": "Explicit Target Groups",

    "Implicit_Personalised_Political_Slurs_NSW": "Implicit Target",
    "Implicit_Hindus_NSW": "Implicit Target",
    "Implicit_Political_Group_Slurs_NSW": "Implicit Target",
    "Implicit_Others_NSW":"Implicit Target",
    "Implicit_Muslims_NSW":"Implicit Target",
    
    "Explicit_Cricket_Personalities_NSW": "Explicit Target Personalities",
    "Explicit_Media_Personalities_NSW": "Explicit Target Personalities",
    "Explicit_Bollywood_Personalities_NSW": "Explicit Target Personalities",
    "Explicit_Historical_Figures_NSW": "Explicit Target Personalities",
    "Explicit_Other_Personalities_NSW": "Explicit Target Personalities",
    "Explicit_Politician_NSW": "Explicit Target Personalities"
}

**Catboost**

In [ ]:
#pip install pytorch-tabnet

In [ ]:
# ---- 1) Reproducibility switches ----
#def set_seed(seed: int = 42):
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic PyTorch
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # In case an op doesn't have a deterministic implementation, don't crash.
    torch.use_deterministic_algorithms(True, warn_only=True)

In [ ]:
SEED = 42
set_seed(SEED)

**CatBoost**

In [ ]:
import catboost as cb
print(cb.__version__)

In [ ]:
def train_evaluate_catboost(X_features,target):
    x_train, x_test, y_train, y_test= train_test_split(X_features, target, stratify=target, test_size=0.2, random_state=8)
    cb_model = cb.CatBoostClassifier(iterations=150,learning_rate=0.01,depth=6,loss_function="Logloss",random_seed=43,eval_metric="Accuracy",l2_leaf_reg=3)
    cb_model.fit(x_train, y_train, early_stopping_rounds=200, use_best_model=True)
    y_pred_cb =cb_model.predict(x_test)
    #y_pred_cb = y_pred_cb.astype(int)
    acc= accuracy_score(y_test,y_pred_cb)
    print("ACCURACY OF THE MODEL:",acc )
    f1=f1_score(y_test,y_pred_cb,average='weighted')
    print(f"F1 score:{f1}")

    # Calculate Precision and Recall
    precision = precision_score(y_test, y_pred_cb, average='weighted')
    recall = recall_score(y_test, y_pred_cb, average='weighted')

    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    return cb_model

In [ ]:
print("Subcategories")
cat_sub=train_evaluate_catboost(X_features_subcat,target)
print("swear and subcategories")
cat_swear=train_evaluate_catboost(X_features_swear,target)
print("Non Swear and subcategories")
cat_nonswear=train_evaluate_catboost(X_features_nonswear,target)

In [ ]:
def plot_catboost_feature_importance(cat_model,X_features,group_map,title):
    
    # --- 1. Get TabNet feature importances ---
    feature_importances_cat=pd.Series(cat_model.get_feature_importance(),cat_model.feature_names_)

    fi = pd.DataFrame({
        "Feature": feature_importances_cat.index,
        "Importance": feature_importances_cat.values
    }).sort_values(by="Importance", ascending=False)


    fi['group'] = fi['Feature'].map(group_map).fillna('Other')

    # --- 3. Choose a colour per group ---
    colour_map = {
        'Explicit Target Groups': '#e6a37a',        # light peach/orange
        'Implicit Target': '#a7c4d9',               # soft blue
        'Explicit Target Personalities': '#c6e1b5', # soft green
    }

    fi['colour'] = fi['group'].map(colour_map)

    plt.figure(figsize=(15, 10))
    plt.barh(fi["Feature"], fi["Importance"],color=fi['colour'])
    plt.gca().invert_yaxis()
    #plt.xlabel("Importance")
    #plt.ylabel("Features")
    plt.title(title,fontweight='bold',fontsize=25,fontname='Times New Roman',color="black")
    

    # Make tick labels bold
    plt.xticks(fontsize=25,fontname='Times New Roman',color="black",fontweight='bold')
    plt.yticks(fontsize=25,fontname='Times New Roman',color="black",fontweight='bold')

    # --- Create legend manually ---
    legend_handles = [
        Patch(facecolor=color, label=group)
        for group, color in colour_map.items()
    ]

    plt.legend(
        handles=legend_handles,
        title="Feature Categories",
        loc="lower right",
        prop={'weight':'bold','size':18,'family':'Times New Roman'},
        title_fontproperties={'weight':'bold','size':18,'family':'Times New Roman'}
    )
    
    plt.savefig(f"Results_2020/{title}.pdf", bbox_inches="tight")
    plt.tight_layout()
    plt.show()
    return feature_importances_cat

In [ ]:
feat_imp_cat_subcat=plot_catboost_feature_importance(cat_sub,X_features_subcat,group_map_sub,"CatBoost Feature Importance for SubCategories")
feat_imp_cat_swear=plot_catboost_feature_importance(cat_swear,X_features_swear,group_map_swear,"CatBoost Feature Importance for Swear Words with Subcategories")
feat_imp_cat_nonswear=plot_catboost_feature_importance(cat_nonswear,X_features_nonswear,group_map_nonswear,"CatBoost Feature Importance for Non Swear Words with Subcategories")

**SHAP for CatBoost**

In [ ]:
import shap

In [ ]:
def shap_plot(model,X_features,target,title):
    x_train, x_test, y_train, y_test= train_test_split(X_features, target, stratify=target, test_size=0.2, random_state=8)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(x_train)
    importances=pd.DataFrame(index=X_features.columns)
    importances['mean_shapley_values'] = np.mean(shap_values, axis=0)
    importances['mean_abs_shapley_values'] = np.mean(np.abs(shap_values),axis=0)
    #shapley = importances.sort_values(by='mean_abs_shapley_values', ascending=False).index
    ## Add to DataFrame
    #shap_features = pd.DataFrame()
    #shap_features['Shapley'] = shapley.values
    # Display
    #shap_features
    shap.summary_plot(shap_values, x_train, plot_type="bar",feature_names=X_features.columns,show=False)
    plt.title(title,fontsize=20, color='black', fontweight='bold',fontname='Times New Roman')
    plt.xlabel("mean(|SHAP value|) (average impact on model output magnitude)", fontsize=20, color='black', fontweight='bold')
    plt.xticks(fontsize=20,fontname='Times New Roman',color="black",fontweight='bold')
    plt.yticks(fontsize=20,fontname='Times New Roman',color="black",fontweight='bold')
    plt.savefig(f"Results_2020/{title}.pdf", bbox_inches="tight")
    plt.tight_layout()
    plt.show()
    #return shap_features
    #return shapley.values
    #return shapley
    return importances['mean_abs_shapley_values']


In [ ]:
shap_subcat=shap_plot(cat_sub,X_features_subcat,target,"SHAP for Subcategories")
shap_swear=shap_plot(cat_swear,X_features_swear,target,"SHAP for Swear Words with Subcategories")
shap_nonswear=shap_plot(cat_nonswear,X_features_nonswear,target,"SHAP for Non Swear Words with Subcategories")

**TabNET**

In [ ]:
import pytorch_tabnet
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score

In [ ]:
def train_evaluate_tabnet(X_features,target):
    # ---- 2) Build model with fixed seed & deterministic device ----
    x_train, x_test, y_train, y_test= train_test_split(X_features.values, target, stratify=target, test_size=0.2, random_state=8)
    tab_clf = TabNetClassifier(
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size": 10, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        verbose=0,   #not to print epochs clutter
        mask_type="entmax",          # or "sparsemax"
        seed=SEED,                   # TabNet's own internal seed
        device_name="cpu"            # use CPU for strict reproducibility
    )
        # ---- Train ----
    tab_clf.fit(x_train,y_train,eval_set=[(x_test, y_test)])
    #tab_clf.fit(x_train,y_train)

    # ---- Predict ----
    y_test_pred = tab_clf.predict(x_test)

    # ---- Metrics ----
    test_acc = accuracy_score(y_test, y_test_pred)

    print("ACCURACY OF THE MODEL:", test_acc)
    f1=f1_score(y_test,y_test_pred,average='weighted')
    print(f"F1 score:{f1}")

    # Calculate Precision and Recall
    precision = precision_score(y_test, y_test_pred, average='weighted')
    recall = recall_score(y_test, y_test_pred, average='weighted')

    print(f'Precision: {precision}')
    print(f'Recall: {recall}')

    return tab_clf

In [ ]:
print("Subcategories")
tab_sub=train_evaluate_tabnet(X_features_subcat,target)
print("swear and subcategories")
tab_swear=train_evaluate_tabnet(X_features_swear,target)
print("Non Swear and subcategories")
tab_nonswear=train_evaluate_tabnet(X_features_nonswear,target)

In [ ]:
def plot_tabnet_feature_importance(tab_model,X_features,group_map,title):
    
    # --- 1. Get TabNet feature importances ---
    feature_importances_tab = pd.Series(tab_model.feature_importances_, index=X_features.columns)


    # Attach group labels
    feat_df = pd.DataFrame({
        'feature': feature_importances_tab.index,
        'importance': feature_importances_tab.values
    })
    feat_df['group'] = feat_df['feature'].map(group_map).fillna('Other')

    # --- 3. Choose a colour per group ---
    colour_map = {
        'Explicit Target Groups': '#e6a37a',        # light peach/orange
        'Implicit Target': '#a7c4d9',               # soft blue
        'Explicit Target Personalities': '#c6e1b5', # soft green
    }

    feat_df['colour'] = feat_df['group'].map(colour_map)

    # --- 4. Sort (ascending or descending) ---
    feat_df = feat_df.sort_values('importance', ascending=True)



    # --- 5. Plot ---
    plt.figure(figsize=(15, 10))
    plt.barh(feat_df['feature'], feat_df['importance'], color=feat_df['colour'])
    #plt.xlabel("Feature Importance")
    #plt.ylabel("Features")
    plt.title(title,fontweight='bold',fontsize=25,fontname='Times New Roman',color="black")
    #plt.title(title,fontsize=20)
    # Make tick labels bold


    plt.xticks(fontsize=25,fontname='Times New Roman',color="black",fontweight='bold')
    plt.yticks(fontsize=25,fontname='Times New Roman',color="black",fontweight='bold')
    # --- Create legend manually ---
    legend_handles = [
        Patch(facecolor=color, label=group)
        for group, color in colour_map.items()
    ]

    plt.legend(
        handles=legend_handles,
        title="Feature Categories",
        loc="lower right",
        prop={'weight':'bold','size':18,'family':'Times New Roman'},
        title_fontproperties={'weight':'bold','size':18,'family':'Times New Roman'}
    )
    plt.savefig(f"Results_2020/{title}.pdf", bbox_inches="tight")
    plt.tight_layout()
    plt.show()
    return feature_importances_tab 

In [ ]:
feat_imp_tab_subcat=plot_tabnet_feature_importance(tab_sub,X_features_subcat,group_map_sub,"TabNet Feature Importance for SubCategories")
feat_imp_tab_swear=plot_tabnet_feature_importance(tab_swear,X_features_swear,group_map_swear,"TabNet Feature Importance for Swear Words with Subcategories")
feat_imp_tab_nonswear=plot_tabnet_feature_importance(tab_nonswear,X_features_nonswear,group_map_nonswear,"TabNet Feature Importance for Non Swear Words with Subcategories")

**Spearman Rank Correlation**

In [ ]:
from scipy.stats import spearmanr

In [ ]:
feat_imp_cat_swear

In [ ]:
feat_imp_tab_swear

In [ ]:
shap_swear

In [ ]:
correl_catshap_subcat,p_catshap_subcat=spearmanr(feat_imp_cat_subcat,shap_subcat)
correl_catshap_swear,p_catshap_swear=spearmanr(feat_imp_cat_swear,shap_swear)
correl_catshap_nonswear,p_catshap_nonswear=spearmanr(feat_imp_cat_nonswear,shap_nonswear)

In [ ]:
correl_tabcat_subcat,p_tabcat_subcat=spearmanr(feat_imp_tab_subcat,feat_imp_cat_subcat)
correl_tabcat_swear,p_tabcat_swear=spearmanr(feat_imp_tab_swear,feat_imp_cat_swear)
correl_tabcat_nonswear,p_tabcat_nonswear=spearmanr(feat_imp_tab_nonswear,feat_imp_cat_nonswear)

In [ ]:
correl_tabshap_subcat,p_tabshap_subcat=spearmanr(feat_imp_tab_subcat,shap_subcat)
correl_tabshap_swear,p_tabshap_swear=spearmanr(feat_imp_tab_swear,shap_swear)
correl_tabshap_nonswear,p_tabshap_nonswear=spearmanr(feat_imp_tab_nonswear,shap_nonswear)

In [ ]:
import pandas as pd

df_all = pd.DataFrame({
    "Comparison": (
        ["CatBoost vs SHAP"] * 3 +
        ["TabNet vs CatBoost"] * 3 +
        ["TabNet vs SHAP"] * 3
    ),
    "Category": ["Subcategories", "Swear Words", "Non-Swear Words"] * 3,
    "Spearman Correlation": [
        correl_catshap_subcat, correl_catshap_swear, correl_catshap_nonswear,
        correl_tabcat_subcat, correl_tabcat_swear, correl_tabcat_nonswear,
        correl_tabshap_subcat, correl_tabshap_swear, correl_tabshap_nonswear
    ],
    "p-value": [
        p_catshap_subcat, p_catshap_swear, p_catshap_nonswear,
        p_tabcat_subcat, p_tabcat_swear, p_tabcat_nonswear,
        p_tabshap_subcat, p_tabshap_swear, p_tabshap_nonswear
    ]
})

pd.options.display.float_format = '{:.4f}'.format
df_all

**Extra Analysis**
kendall tau for extra analysis

**Kendall tau**

In [ ]:
from scipy.stats import kendalltau

In [ ]:
kendall_catshap_subcat,kp_catshap_subcat=kendalltau(feat_imp_cat_subcat,shap_subcat)
kendall_catshap_swear,kp_catshap_swear=kendalltau(feat_imp_cat_swear,shap_swear)
kendall_catshap_nonswear,kp_catshap_nonswear=kendalltau(feat_imp_cat_nonswear,shap_nonswear)

In [ ]:
kendall_tabcat_subcat,kp_tabcat_subcat=kendalltau(feat_imp_tab_subcat,feat_imp_cat_subcat)
kendall_tabcat_swear,kp_tabcat_swear=kendalltau(feat_imp_tab_swear,feat_imp_cat_swear)
kendall_tabcat_nonswear,kp_tabcat_nonswear=kendalltau(feat_imp_tab_nonswear,feat_imp_cat_nonswear)

In [ ]:
kendall_tabshap_subcat,kp_tabshap_subcat=kendalltau(feat_imp_tab_subcat,shap_subcat)
kendall_tabshap_swear,kp_tabshap_swear=kendalltau(feat_imp_tab_swear,shap_swear)
kendall_tabshap_nonswear,kp_tabshap_nonswear=kendalltau(feat_imp_tab_nonswear,shap_nonswear)

In [ ]:
import pandas as pd

df_all = pd.DataFrame({
    "Comparison": (
        ["CatBoost vs SHAP"] * 3 +
        ["TabNet vs CatBoost"] * 3 +
        ["TabNet vs SHAP"] * 3
    ),
    "Category": ["Subcategories", "Swear Words", "Non-Swear Words"] * 3,
    "kendalltau": [
        kendall_catshap_subcat, kendall_catshap_swear, kendall_catshap_nonswear,
        kendall_tabcat_subcat, kendall_tabcat_swear, kendall_tabcat_nonswear,
        kendall_tabshap_subcat, kendall_tabshap_swear, kendall_tabshap_nonswear
    ],
    "p-value": [
        kp_catshap_subcat, kp_catshap_swear, kp_catshap_nonswear,
        kp_tabcat_subcat, kp_tabcat_swear, kp_tabcat_nonswear,
       kp_tabshap_subcat, kp_tabshap_swear, kp_tabshap_nonswear
    ]
})

#pd.options.display.float_format = '{:.4f}'.format
pd.options.display.float_format = '{:.4f}'.format
df_all

In [ ]:
#print(np.array_equal(feat_imp_tab_nonswear, feat_imp_tab_swear))
#print(np.array_equal(feat_imp_cat_nonswear, shap_swear))

Kendall tau is for extra analysis

In [ ]:
print("Done")